# rxn_core Staged Pipeline Tutorial

This notebook walks through the three resumable stages and the composed full pipeline:

1. Stage 1: R-P alignment and mechanism discovery
2. Stage 2: TS / IG / GT verification and scoring
3. Stage 3: viewer and report generation
4. Full pipeline: run all stages in one call

The heavy cells are guarded by `RUN_EXAMPLE = False` by default. Set it to `True` after choosing a real cached step.

## 0. Environment And Parameters

Expected cached step layout:

```text
<BGCP_WORK>/<step>/
  R/                    reactant-complex xyz + wbo
  P/                    product-complex xyz + wbo
  sp_iter<N>/           optional IG xyz + wbo
  hess_iter<N>/         optional IG g98.out
  sp_groundtruth/       optional GT xyz + wbo
  hess_groundtruth/     optional GT g98.out
```

Use `--xtb-mode cache-only` or set `pipeline.XTB_CACHE_MODE = "cache-only"` to guarantee the notebook does not launch xtb.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT = Path.cwd()
if (PROJECT / "src").exists():
    sys.path.insert(0, str(PROJECT / "src"))

import rxn_core.pipeline as pipeline
from rxn_core.pipeline import (
    load_step_inputs,
    load_ts_targets,
    run_rp_stage,
    run_ts_stage,
    write_view_stage,
    write_rp_alignment_files,
    write_ts_alignment_files,
    write_stage_json,
    read_stage_json,
    pipeline_stage_paths,
    step_inputs_from_arrays,
    ts_target_from_arrays,
    rp_stage_config,
    ts_stage_config,
)

# Change these for your run.
STEP = os.environ.get("RXN_CORE_TUTORIAL_STEP", "pr1.tempo_ts4")
RUN_EXAMPLE = False
RUN_FULL_EXAMPLE = False
INCLUDE_GT = True
SAVE_ALIGNMENT_FILES = True
INNER_WORKERS = 4
MECHANISM_IDS = None  # Example: [1] or [2]

WORK = Path(os.environ.get("BGCP_WORK", PROJECT / "data" / "xtb_frequency_calculations"))
OUT_ROOT = PROJECT / "out" / "tutorial_views"
STAGE_ROOT = PROJECT / "out" / "tutorial_stages"
ALIGNMENT_OUT_ROOT = PROJECT / "out" / "tutorial_alignments"

# Configure the pipeline module used by the public helpers.
pipeline.WORK = WORK
pipeline.OUT_ROOT = OUT_ROOT
pipeline.STAGE_ROOT = STAGE_ROOT
pipeline.ALIGNMENT_OUT_ROOT = ALIGNMENT_OUT_ROOT
pipeline.INCLUDE_GT = INCLUDE_GT
pipeline.XTB_CACHE_MODE = "cache-only"

print("STEP:", STEP)
print("WORK:", WORK)
print("RUN_EXAMPLE:", RUN_EXAMPLE)

In [ ]:
def show_json(path, max_chars=2000):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path}")
        return None
    data = json.loads(path.read_text())
    text = json.dumps(data, indent=2)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))
    return data

def summarize_mechanisms(result):
    for mech in result.get("mechanisms", []):
        print(
            f"mechanism {mech['id']}: broken={mech.get('broken_bonds_R')} "
            f"formed={mech.get('formed_bonds_R')} core={mech.get('core_atoms')}"
        )

def print_cli(cmd):
    print("\\\n".join(cmd))

## 1. Stage 1: R-P Alignment And Mechanism Discovery

Stage 1 runs R-P sweep cut, chooses minimum broken/forming-bond mechanisms, and writes a reusable `rp_stage.json` artifact.

In [ ]:
print_cli([
    "rxn-core --stage rp --steps <step>",
    "  --workers 8 --parallel-mode inner --inner-workers 8",
    "  --xtb-mode cache-only",
])

print_cli([
    "rxn-core --stage rp --steps <step>",
    "  --save-alignment-files",
    "  --xtb-mode cache-only",
])

In [ ]:
RP_CONFIG = rp_stage_config()
# Edit these if you want to test sensitivity.
# RP_CONFIG["iso_tol"] = 1.0
# RP_CONFIG["dwbo_threshold"] = 0.5

if RUN_EXAMPLE:
    inputs = load_step_inputs(STEP)
    rp = run_rp_stage(inputs, config=RP_CONFIG, inner_workers=INNER_WORKERS)
    paths = pipeline_stage_paths(STEP)
    write_stage_json(paths.rp_json, rp)
    print("wrote", paths.rp_json)
    summarize_mechanisms(rp)
    if SAVE_ALIGNMENT_FILES:
        rp_files = write_rp_alignment_files(inputs, rp)
        print("wrote", rp_files["manifest"])
else:
    print("Set RUN_EXAMPLE = True to run Stage 1 on STEP.")

Stage 1 coordinate export layout:

```text
out/bgcp_alignments/<step>/
  manifest.json
  R.xyz
  P_original.xyz
  mechanisms/mechanism_001/
    R.xyz
    P_aligned.xyz
    neb_endpoints.xyz
    mapping_R_to_P.csv
    mechanism.json
```

`P_aligned.xyz` is product geometry reindexed into R atom order. No Kabsch/spatial fitting is used.

In [ ]:
rp_path = STAGE_ROOT / STEP / "rp_stage.json"
if rp_path.exists():
    rp_preview = show_json(rp_path, max_chars=2500)
else:
    print("No rp_stage.json yet:", rp_path)

## 2. Stage 2: TS / IG / GT Verification

Stage 2 resumes from `rp_stage.json`, aligns each GT/IG/TS target through the selected mechanism core, and scores the selected imaginary mode.

In [ ]:
print_cli([
    "rxn-core --stage ts --steps <step>",
    "  --include-gt",
    "  --workers 8 --parallel-mode inner --inner-workers 8",
    "  --xtb-mode cache-only",
])

print_cli([
    "rxn-core --stage ts --steps <step>",
    "  --mechanism 2 --include-gt --save-alignment-files",
    "  --xtb-mode cache-only",
])

In [ ]:
TS_CONFIG = ts_stage_config()
# Edit score hypotheses here if needed.
# TS_CONFIG["score"]["W_RXN"] = 1.0
# TS_CONFIG["score"]["W_CORE"] = 0.2

if RUN_EXAMPLE:
    if "inputs" not in globals():
        inputs = load_step_inputs(STEP)
    if "rp" not in globals():
        rp = read_stage_json(STAGE_ROOT / STEP / "rp_stage.json")
    targets = load_ts_targets(inputs, include_gt=INCLUDE_GT)
    print("targets:", [(t.kind, t.label) for t in targets])
    ts = run_ts_stage(
        inputs,
        rp,
        targets,
        config=TS_CONFIG,
        inner_workers=INNER_WORKERS,
        mechanism_ids=MECHANISM_IDS,
    )
    paths = pipeline_stage_paths(STEP)
    write_stage_json(paths.ts_json, ts)
    print("wrote", paths.ts_json)
    if SAVE_ALIGNMENT_FILES:
        ts_files = write_ts_alignment_files(inputs, ts)
        print("wrote", ts_files["manifest"])
else:
    print("Set RUN_EXAMPLE = True to run Stage 2 on STEP.")

Stage 2 coordinate export layout:

```text
out/bgcp_alignments/<step>/ts_alignments/
  manifest.json
  mechanisms/mechanism_002/gt_GT/
    TS_native.xyz
    TS_core_aligned_R_frame.xyz
    picked_mode_R_frame.xyz
    score.json
```

Stage 2 scores core mappings, not full spectator bijections. `TS_core_aligned_R_frame.xyz` materializes the selected best-S core mapping in R atom order: mapped core atoms come from the target, while spectators remain at the R endpoint.

In [ ]:
ts_path = STAGE_ROOT / STEP / "ts_stage.json"
if ts_path.exists():
    ts_preview = show_json(ts_path, max_chars=2500)
else:
    print("No ts_stage.json yet:", ts_path)

## 3. Stage 3: Viewer And Report Generation

Stage 3 is presentation-only. It reloads stage artifacts and writes `view.html` plus `_eval_slim.json`.

In [ ]:
print_cli([
    "rxn-core --stage view --steps <step>",
    "  --include-gt",
])

In [ ]:
if RUN_EXAMPLE:
    if "inputs" not in globals():
        inputs = load_step_inputs(STEP)
    if "rp" not in globals():
        rp = read_stage_json(STAGE_ROOT / STEP / "rp_stage.json")
    if "ts" not in globals() and (STAGE_ROOT / STEP / "ts_stage.json").exists():
        ts = read_stage_json(STAGE_ROOT / STEP / "ts_stage.json")
    view = write_view_stage(inputs, rp, globals().get("ts"), include_gt=INCLUDE_GT)
    print("view:", view["view_html"])
    print("eval:", view["eval_slim_json"])
else:
    print("Set RUN_EXAMPLE = True to regenerate the viewer.")

## 4. Full Pipeline

The full pipeline composes Stage 1, Stage 2, and Stage 3 in one call.

In [ ]:
print_cli([
    "rxn-core --stage full --steps <step>",
    "  --include-gt --save-alignment-files",
    "  --workers 8 --parallel-mode inner --inner-workers 8",
    "  --xtb-mode cache-only",
])

In [ ]:
if RUN_FULL_EXAMPLE:
    result = pipeline.run_full_pipeline_stage(
        STEP,
        inner_workers=INNER_WORKERS,
        mechanism_ids=MECHANISM_IDS,
        save_alignment_files=SAVE_ALIGNMENT_FILES,
    )
    print("view:", result["view"]["view_html"])
else:
    print("Set RUN_FULL_EXAMPLE = True to run the composed full pipeline.")

## 5. Tiny In-Memory Example

This example does not need xtb or cached files. It runs the R-P mechanism-discovery API on two H2 graphs to show the array-based entry point.

In [ ]:
import numpy as np

el = ["H", "H"]
xyz_r = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.75]])
xyz_p = np.array([[1.0, 0.0, 0.0], [1.0, 0.0, 0.75]])
wbo = np.array([[0.0, 0.9], [0.9, 0.0]])

small_inputs = step_inputs_from_arrays("h2_demo", el, xyz_r, wbo, el, xyz_p, wbo)
small_cfg = rp_stage_config()
small_cfg["n_seeds"] = 1
small_rp = run_rp_stage(small_inputs, config=small_cfg, inner_workers=0)

summarize_mechanisms(small_rp)
print(json.dumps(small_rp["mechanisms"][0], indent=2)[:1500])

## 6. Common Runtime Knobs

Parallelism:

```text
--workers
--parallel-mode auto|outer|inner
--inner-workers
--auto-inner-workers
```

Cache and spin state:

```text
--xtb-mode auto|cache-only
--xtb-omp-threads
--xtb-max-threads
--charge
--multiplicity
```

Algorithm and scoring hypotheses:

```text
--iso-tol
--dwbo-threshold
--symmetry-wbo-tol
--w-rxn
--w-core
--imag-pen
```